# HW 6 Ditto per Entity Matching (FAIR-DA4ER)

Questo notebook implementa il punto **4.G** e **4.H** dell'Homework 6 utilizzando la repository ufficiale:
**https://github.com/MarcoNapoleone/FAIR-DA4ER**

### Caratteristiche Principali:
1. **Ambiente e Dipendenze**: Clone di `FAIR-DA4ER`, installazione requirements, download stopwords e supporto per `matcher.py`.
2. **Caricamento Artifact**: Lettura dei dataset e delle coppie etichettate (costruiti senza VIN in `HW_6_GT_R_2.ipynb`).
3. **Candidate Generation B1 e B2**: Blocco esatto su `make` (B1) e su `year` (B2).
4. **Serializzazione Standard Ditto**: Conversione dei record nel formato `COL <attr> VAL <val> ... \t COL <attr> VAL <val> ... \t <label>` (`train.tsv`, `valid.tsv`, `test.tsv`).
5. **Configurazione Ufficiale in `configs.json`**: Registrazione formale del task `UsedCars` con puntamento esplicito a `trainset`, `validset` e `testset`.
6. **Addestramento con `train_ditto.py` (Punto 4.G)**: Fine-tuning di `roberta-base` con mixed precision (FP16) e salvataggio automatico del miglior checkpoint.
7. **Selezione della Soglia su Validation ($th^*$)**: Ottimizzazione della soglia decisionale $th^*$ esclusivamente su validation (zero data leakage sul test).
8. **Sanity Check con `matcher.py`**: Dimostrazione empirica dell'equivalenza esatta tra lo script `matcher.py` ufficiale e la funzione di inferenza vettorizzata.
9. **Misurazione Omogenea dell'Inferenza (Punto 4.H)**: Scomposizione rigorosa in $T_{\text{blocking}}$ e $T_{\text{matching}}$ in-memory (con `torch.autocast('cuda', dtype=torch.float16)` e `torch.cuda.synchronize()`).
10. **Tabella di Sintesi Comparativa Finale**: Confronto completo delle 6 pipeline (`B1/B2 - RecordLinkage`, `B1/B2 - Dedupe`, `B1/B2 - Ditto`).


##Runtime, Dipendenze e Clone del Repository


In [1]:
%pip install -q -U huggingface-hub "transformers>=4.40" "accelerate>=0.26" "scikit-learn>=1.3" pyarrow tensorboardX loguru jsonlines recordlinkage

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 798.3/798.3 kB 58.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 145.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 162.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.9/926.9 kB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.4/360.4 kB 38.2 MB/s eta 0:00:00


In [2]:
import os
import sys
import time
import subprocess
from pathlib import Path
from loguru import logger
from google.colab import drive

os.environ['TZ'] = 'Europe/Rome'
time.tzset()

logger.remove()
logger.add(
    sys.stdout,
    colorize=True,
    format="<green>{time:YYYY-MM-DD HH:mm:ss}</green> | <level>{level: <8}</level> | <cyan>{message}</cyan>"
)

drive.mount('/content/drive', force_remount=True)

#clone della repository ufficiale FAIR-DA4ER
FAIR_REPO_URL = 'https://github.com/MarcoNapoleone/FAIR-DA4ER.git'
FAIR_REPO_DIR = Path('/content/FAIR-DA4ER')
if not FAIR_REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--depth', '1', FAIR_REPO_URL, str(FAIR_REPO_DIR)], check=True)
    logger.success(f"Repository clonata con successo in: {FAIR_REPO_DIR}")
else:
    logger.info(f"Repository già presente: {FAIR_REPO_DIR}")

#download stopwords di Ditto
subprocess.run(['python', str(FAIR_REPO_DIR / 'ditto' / 'download_stopwords.py')], check=True)

REPO_DITTO_DIR = FAIR_REPO_DIR / 'ditto'
if str(REPO_DITTO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DITTO_DIR))
os.chdir(str(REPO_DITTO_DIR))
assert (REPO_DITTO_DIR / 'train_ditto.py').exists(), 'train_ditto.py non trovato!'
assert (REPO_DITTO_DIR / 'configs.json').exists(), 'configs.json non trovato!'

DRIVE_DATA_DIR = '/content/drive/MyDrive/data'
if not os.path.exists(DRIVE_DATA_DIR) and os.path.exists('/content/drive/MyDrive/ing_dati/data'):
    DRIVE_DATA_DIR = '/content/drive/MyDrive/ing_dati/data'

DITTO_DATA_DIR = REPO_DITTO_DIR / 'data' / 'used_cars'
DITTO_DATA_DIR.mkdir(parents=True, exist_ok=True)
DITTO_DRIVE_BACKUP = f'{DRIVE_DATA_DIR}/ditto_used_cars'
os.makedirs(DITTO_DRIVE_BACKUP, exist_ok=True)

logger.info(f"Repository Ditto : {FAIR_REPO_DIR}")
logger.info(f"Data dir locale  : {DITTO_DATA_DIR}")
logger.info(f"Drive data input : {DRIVE_DATA_DIR}")


Mounted at /content/drive
2026-09-14 15:42:29 | SUCCESS  | Repository clonata con successo in: /content/FAIR-DA4ER
2026-09-14 15:42:33 | INFO     | Repository Ditto : /content/FAIR-DA4ER
2026-09-14 15:42:33 | INFO     | Data dir locale  : /content/FAIR-DA4ER/ditto/data/used_cars
2026-09-14 15:42:33 | INFO     | Drive data input : /content/drive/MyDrive/data


### Patch di compatibilita' per `apex` in `matcher.py`

`matcher.py` della repository fa `from apex import amp` a livello di modulo (riga usata da `load_model()`
per il fine-tuning fp16 in stile Apex, oggi superato da `torch.cuda.amp`). `apex` e' una libreria NVIDIA
legacy non installata negli ambienti moderni, quindi l'import fallirebbe subito.

Invece di iniettare un modulo finto a runtime (soluzione fragile, da ripetere ad ogni sessione),
patchiamo **una sola volta** il file `matcher.py` appena clonato, rendendo `apex` opzionale con un
fallback no-op nativo. Lo script CLI risulta cosi' funzionante al 100% anche in ambienti senza apex,
senza bisogno di alcun mock nelle celle successive.


In [3]:
#patch una tantum di matcher.py: rende apex opzionale con fallback no-op e supporta sia .txt che .tsv
MATCHER_PY_PATH = REPO_DITTO_DIR / 'matcher.py'
matcher_src = MATCHER_PY_PATH.read_text(encoding='utf-8')

OLD_IMPORT = 'from apex import amp'
NEW_IMPORT = (
    'try:\n'
    '    from apex import amp\n'
    'except ImportError:\n'
    '    class _NoOpAmp:\n'
    '        """Fallback no-op quando apex non e\' disponibile (ambienti moderni senza NVIDIA apex).\n'
    '        matcher.py lo invoca solo per il fine-tuning fp16 in stile Apex; il nostro flusso usa\n'
    '        gia\' torch.autocast, quindi initialize() puo\' restituire il modello invariato."""\n'
    '        @staticmethod\n'
    '        def initialize(model, *args, **kwargs):\n'
    '            return model\n'
    '    amp = _NoOpAmp()'
)

if OLD_IMPORT in matcher_src and 'try:\n    from apex' not in matcher_src:
    matcher_src = matcher_src.replace(OLD_IMPORT, NEW_IMPORT, 1)

if "if '.txt' in input_path:" in matcher_src and "or '.tsv' in input_path" not in matcher_src:
    matcher_src = matcher_src.replace(
        "if '.txt' in input_path:",
        "if '.txt' in input_path or '.tsv' in input_path:"
    )

MATCHER_PY_PATH.write_text(matcher_src, encoding='utf-8')
logger.success(f"matcher.py patchato per apex e compatibilita' estensioni ({MATCHER_PY_PATH})")


2026-09-14 15:42:33 | SUCCESS  | matcher.py patchato per apex e compatibilita' estensioni (/content/FAIR-DA4ER/ditto/matcher.py)


##Configurazione Riproducibile e Device


In [4]:
import random
import time
import copy
import json
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import precision_recall_fscore_support

SEED = 42

def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
logger.info(f"Device in uso: {device}")
if device.type != 'cuda':
    logger.warning("ATTENZIONE: Stai eseguendo su CPU! Per il training di RoBERTa seleziona Runtime: T4/L4 GPU.")

2026-09-14 15:42:39 | INFO     | Device in uso: cuda


##Caricamento degli Artifact da Google Drive (Privi di VIN)


In [5]:
required_files = [
    'vehicles_train.parquet', 'used_cars_train.parquet',
    'vehicles_validation.parquet', 'used_cars_validation.parquet',
    'vehicles_test.parquet', 'used_cars_test.parquet',
    'train_gt.csv', 'validation_gt.csv', 'test_gt.csv',
    'train_pairs_1_5.csv', 'val_pairs_1_5.csv'
]
missing = [name for name in required_files if not os.path.exists(f'{DRIVE_DATA_DIR}/{name}')]
assert not missing, f'Mancano questi artifact in {DRIVE_DATA_DIR}: {", ".join(missing)}'

vehicles_train = pd.read_parquet(f'{DRIVE_DATA_DIR}/vehicles_train.parquet')
used_cars_train = pd.read_parquet(f'{DRIVE_DATA_DIR}/used_cars_train.parquet')
vehicles_val = pd.read_parquet(f'{DRIVE_DATA_DIR}/vehicles_validation.parquet')
used_cars_val = pd.read_parquet(f'{DRIVE_DATA_DIR}/used_cars_validation.parquet')
vehicles_test = pd.read_parquet(f'{DRIVE_DATA_DIR}/vehicles_test.parquet')
used_cars_test = pd.read_parquet(f'{DRIVE_DATA_DIR}/used_cars_test.parquet')

train_gt = pd.read_csv(f'{DRIVE_DATA_DIR}/train_gt.csv')
validation_gt = pd.read_csv(f'{DRIVE_DATA_DIR}/validation_gt.csv')
test_gt = pd.read_csv(f'{DRIVE_DATA_DIR}/test_gt.csv')
train_pairs = pd.read_csv(f'{DRIVE_DATA_DIR}/train_pairs_1_5.csv')
val_pairs = pd.read_csv(f'{DRIVE_DATA_DIR}/val_pairs_1_5.csv')

for frame in [vehicles_train, used_cars_train, vehicles_val, used_cars_val, vehicles_test, used_cars_test]:
    assert 'vin' not in frame.columns, 'Violazione consegna 4.B: il VIN non deve essere presente nei dataset!'

logger.info(f"Train records : {len(vehicles_train):,} / {len(used_cars_train):,} | coppie 1:5: {len(train_pairs):,}")
logger.info(f"Val records   : {len(vehicles_val):,} / {len(used_cars_val):,} | coppie 1:5: {len(val_pairs):,}")
logger.info(f"Test records  : {len(vehicles_test):,} / {len(used_cars_test):,} | true matches: {len(test_gt):,}")

2026-09-14 15:43:00 | INFO     | Train records : 2,368 / 2,368 | coppie 1:5: 14,208
2026-09-14 15:43:00 | INFO     | Val records   : 789 / 789 | coppie 1:5: 4,734
2026-09-14 15:43:00 | INFO     | Test records  : 790 / 790 | true matches: 790


##Candidate Generation per Blocking B1 e B2


In [6]:
import recordlinkage

LEFT_ID = 'orig_cl_id'
RIGHT_ID = 'orig_uc_id'

def candidates_from_blocking(left_df, right_df, strategy):
    """
    genera le coppie candidate tramite la libreria recordlinkage in modo:
    - "make": blocco esatto su marca (B1, ~42.494 coppie)
    - "year": Sorted Neighbourhood su anno con finestra 1 (B2, ~46.870 coppie)
    """
    indexer = recordlinkage.Index()
    left_indexed = left_df.set_index(LEFT_ID) if left_df.index.name != LEFT_ID else left_df
    right_indexed = right_df.set_index(RIGHT_ID) if right_df.index.name != RIGHT_ID else right_df

    if strategy == 'make':
        indexer.block(left_on='make', right_on='make')
        pairs = indexer.index(left_indexed, right_indexed)
    elif strategy == 'year':
        left_temp = left_indexed.copy()
        right_temp = right_indexed.copy()
        left_temp['year'] = left_temp['year'].astype(str)
        right_temp['year'] = right_temp['year'].astype(str)
        indexer.sortedneighbourhood(left_on='year', right_on='year', window=1)
        pairs = indexer.index(left_temp, right_temp)
    else:
        raise ValueError(f"Strategia blocking non supportata: {strategy}")

    return pd.DataFrame(list(pairs), columns=[LEFT_ID, RIGHT_ID]).drop_duplicates().reset_index(drop=True)

def pairs_with_labels(candidate_pairs, gt_df):
    """assegna la label binaria 1/0 confrontando con la ground truth del rispettivo split."""
    truth = set(gt_df[[LEFT_ID, RIGHT_ID]].itertuples(index=False, name=None))
    df = candidate_pairs.copy()
    df['label'] = [
        int(pair in truth)
        for pair in df[[LEFT_ID, RIGHT_ID]].itertuples(index=False, name=None)
    ]
    return df

val_b1_cand = candidates_from_blocking(vehicles_val, used_cars_val, 'make')
val_b2_cand = candidates_from_blocking(vehicles_val, used_cars_val, 'year')
test_b1_cand = candidates_from_blocking(vehicles_test, used_cars_test, 'make')
test_b2_cand = candidates_from_blocking(vehicles_test, used_cars_test, 'year')

val_b1_pairs = pairs_with_labels(val_b1_cand, validation_gt)
val_b2_pairs = pairs_with_labels(val_b2_cand, validation_gt)
test_b1_pairs = pairs_with_labels(test_b1_cand, test_gt)
test_b2_pairs = pairs_with_labels(test_b2_cand, test_gt)

def log_blocking_stats(name, pairs, gt):
    captured = int(pairs['label'].sum())
    recall = captured / len(gt) if len(gt) else 0.0
    logger.info(f"{name:25s} | Candidati: {len(pairs):6,d} | Match catturati: {captured:4d}/{len(gt):4d} | Blocking Recall: {recall:.4f}")

logger.info("STATISTICHE BLOCKING:")
log_blocking_stats('Validation B1 (make)', val_b1_pairs, validation_gt)
log_blocking_stats('Validation B2 (year)', val_b2_pairs, validation_gt)
log_blocking_stats('Test B1 (make)', test_b1_pairs, test_gt)
log_blocking_stats('Test B2 (year)', test_b2_pairs, test_gt)

2026-09-14 15:43:00 | INFO     | STATISTICHE BLOCKING:
2026-09-14 15:43:00 | INFO     | Validation B1 (make)      | Candidati: 45,548 | Match catturati:  764/ 789 | Blocking Recall: 0.9683
2026-09-14 15:43:00 | INFO     | Validation B2 (year)      | Candidati: 45,771 | Match catturati:  783/ 789 | Blocking Recall: 0.9924
2026-09-14 15:43:00 | INFO     | Test B1 (make)            | Candidati: 43,456 | Match catturati:  778/ 790 | Blocking Recall: 0.9848
2026-09-14 15:43:00 | INFO     | Test B2 (year)            | Candidati: 49,757 | Match catturati:  787/ 790 | Blocking Recall: 0.9962


##Serializzazione dei Record nel Formato Ufficiale Ditto

Ogni riga serializzata per Ditto è costituita da tre campi separati da tabulazione:
`record_sinistro \t record_destro \t label`
dove ciascun record ha il formato:
`COL nome_attributo VAL valore COL ...`

I file `train.tsv`, `valid.tsv` e `test.tsv` vengono scritti in `FAIR-DA4ER/ditto/data/used_cars/` per la registrazione formale e il training.


In [7]:
DITTO_COLUMNS = [
    'make', 'model', 'year', 'price', 'mileage', 'fuel',
    'transmission', 'body_type', 'drivetrain', 'cylinders', 'color'
]

def serialize_record(record_series):
    """converte una riga Pandas in stringa serializzata Ditto: COL col VAL val ..."""
    tokens = []
    for col, val in record_series.items():
        if pd.notna(val):
            val_str = str(val).replace('\n', ' ').replace('\t', ' ').strip()
            if val_str and val_str.lower() != '<na>':
                tokens.append(f'COL {col} VAL {val_str}')
    return ' '.join(tokens)

def build_serialization_cache(df, id_col):
    """
    calcola una sola volta la serializzazione Ditto per ciascun record univoco (one-off cache),
    evitando ricalcoli ridondanti tramite itertuples() ad alte prestazioni.
    """
    cache = {}
    for row in df[[id_col] + DITTO_COLUMNS].itertuples(index=False):
        rec_id = row[0]
        tokens = []
        for col_name, val in zip(DITTO_COLUMNS, row[1:]):
            if pd.notna(val):
                val_str = str(val).replace('\n', ' ').replace('\t', ' ').strip()
                if val_str and val_str.lower() != '<na>':
                    tokens.append(f'COL {col_name} VAL {val_str}')
        cache[rec_id] = ' '.join(tokens)
    return cache

def write_ditto_tsv(pairs_df, left_cache, right_cache, output_path):
    """serializza e scrive su disco un file TSV per Ditto usando le cache precalcolate dei record."""
    out_path = Path(output_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, 'w', encoding='utf-8') as f:
        for row in pairs_df[[LEFT_ID, RIGHT_ID, 'label']].itertuples(index=False):
            left_txt = left_cache[row[0]]
            right_txt = right_cache[row[1]]
            f.write(f'{left_txt}\t{right_txt}\t{int(row[2])}\n')
    logger.info(f"Salvate {len(pairs_df):,d} coppie in: {output_path}")

# precalcolo delle cache di serializzazione per gli split (costo one-off condiviso)
train_left_ser = build_serialization_cache(vehicles_train, LEFT_ID)
train_right_ser = build_serialization_cache(used_cars_train, RIGHT_ID)

val_left_ser = build_serialization_cache(vehicles_val, LEFT_ID)
val_right_ser = build_serialization_cache(used_cars_val, RIGHT_ID)

test_left_ser = build_serialization_cache(vehicles_test, LEFT_ID)
test_right_ser = build_serialization_cache(used_cars_test, RIGHT_ID)

# dataset per il training del modello (split con rapporto 1:5)
train_tsv_path = str(DITTO_DATA_DIR / 'train.tsv')
write_ditto_tsv(train_pairs, train_left_ser, train_right_ser, train_tsv_path)

#dataset per la validazione durante il training
val_tsv_path = str(DITTO_DATA_DIR / 'valid.tsv')
write_ditto_tsv(val_pairs, val_left_ser, val_right_ser, val_tsv_path)

#dataset di test locale principale (test.tsv)
test_tsv_path = str(DITTO_DATA_DIR / 'test.tsv')
write_ditto_tsv(test_b1_pairs, test_left_ser, test_right_ser, test_tsv_path)

#file di test specifici per i task simmetrici registrati in configs.json
test_b1_path = str(DITTO_DATA_DIR / 'test_b1_make.tsv')
write_ditto_tsv(test_b1_pairs, test_left_ser, test_right_ser, test_b1_path)

test_b2_path = str(DITTO_DATA_DIR / 'test_b2_year.tsv')
write_ditto_tsv(test_b2_pairs, test_left_ser, test_right_ser, test_b2_path)

#backup su Google Drive
write_ditto_tsv(train_pairs, train_left_ser, train_right_ser, f'{DITTO_DRIVE_BACKUP}/train.tsv')
write_ditto_tsv(val_pairs, val_left_ser, val_right_ser, f'{DITTO_DRIVE_BACKUP}/valid.tsv')
write_ditto_tsv(test_b1_pairs, test_left_ser, test_right_ser, f'{DITTO_DRIVE_BACKUP}/test_b1_make.tsv')
write_ditto_tsv(test_b2_pairs, test_left_ser, test_right_ser, f'{DITTO_DRIVE_BACKUP}/test_b2_year.tsv')

2026-09-14 15:43:00 | INFO     | Salvate 14,208 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/train.tsv
2026-09-14 15:43:00 | INFO     | Salvate 4,734 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/valid.tsv
2026-09-14 15:43:00 | INFO     | Salvate 43,456 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/test.tsv
2026-09-14 15:43:00 | INFO     | Salvate 43,456 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/test_b1_make.tsv
2026-09-14 15:43:00 | INFO     | Salvate 49,757 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/test_b2_year.tsv
2026-09-14 15:43:02 | INFO     | Salvate 14,208 coppie in: /content/drive/MyDrive/data/ditto_used_cars/train.tsv
2026-09-14 15:43:03 | INFO     | Salvate 4,734 coppie in: /content/drive/MyDrive/data/ditto_used_cars/valid.tsv
2026-09-14 15:43:04 | INFO     | Salvate 43,456 coppie in: /content/drive/MyDrive/data/ditto_used_cars/test_b1_make.tsv
2026-09-14 15:43:05 | INFO     | Salvate 49,757 coppie in: /content/drive/MyDrive/data/ditto_

In [8]:
def audit_tsv(path):
    labels = []
    with open(path, encoding='utf-8') as f:
        for num, line in enumerate(f, start=1):
            parts = line.rstrip('\n').split('\t')
            assert len(parts) == 3, f'Riga {num}: attesi 3 campi, trovati {len(parts)}'
            assert parts[2] in {'0', '1'}, f'Riga {num}: label non valida {parts[2]}'
            labels.append(int(parts[2]))
    return {'totale_righe': len(labels), 'positivi': sum(labels), 'negativi': len(labels) - sum(labels)}

audit_df = pd.DataFrame({
    'train.tsv': audit_tsv(train_tsv_path),
    'valid.tsv': audit_tsv(val_tsv_path),
    'test.tsv': audit_tsv(test_tsv_path)
}).T
display(audit_df)

with open(train_tsv_path, encoding='utf-8') as f:
    first_left, first_right, first_lbl = f.readline().rstrip('\n').split('\t')
logger.info(f"Esempio serializzazione record sinistro: {first_left[:120]} ...")
logger.info(f"Esempio serializzazione record destro  : {first_right[:120]} ...")
logger.info(f"Label: {first_lbl}")

,totale_righe,positivi,negativi
train.tsv,14208,2368,11840
valid.tsv,4734,789,3945
test.tsv,43456,778,42678


2026-09-14 15:43:05 | INFO     | Esempio serializzazione record sinistro: COL model VAL grand cherokee suv COL year VAL 2020 COL price VAL 29466 COL mileage VAL 14873.0 COL fuel VAL gas COL tran ...
2026-09-14 15:43:05 | INFO     | Esempio serializzazione record destro  : COL make VAL jeep COL model VAL grand cherokee COL year VAL 2020 COL price VAL 30499.0 COL mileage VAL 14873.0 COL fuel  ...
2026-09-14 15:43:05 | INFO     | Label: 1


## Configurazione Ufficiale in `configs.json` (FAIR-DA4ER)

Come previsto dall'architettura della repository `FAIR-DA4ER`, registriamo i dataset dedicati nel file centrale `ditto/configs.json` specificando i percorsi di `trainset`, `validset` e `testset`.

Registriamo in modo trasparente e simmetrico:
- `UsedCars`: configurazione master per il training del task automobilistico su dataset 1:5;
- `UsedCars_B1`: configurazione specifica per la valutazione del blocking B1 (Make) su `test_b1_make.tsv`;
- `UsedCars_B2`: configurazione specifica per la valutazione del blocking B2 (Year $\pm 1$) su `test_b2_year.tsv`.



In [9]:
CONFIGS_PATH = REPO_DITTO_DIR / 'configs.json'

with open(CONFIGS_PATH, 'r', encoding='utf-8') as f:
    all_configs = json.load(f)

# Rimozione di eventuali entry precedenti per garantire piena idempotenza
tasks_to_remove = {'UsedCars', 'UsedCars_B1', 'UsedCars_B2'}
all_configs = [c for c in all_configs if c.get('name') not in tasks_to_remove]

used_cars_config = {
    'name': 'UsedCars',
    'task_type': 'classification',
    'vocab': ['0', '1'],
    'trainset': 'data/used_cars/train.tsv',
    'validset': 'data/used_cars/valid.tsv',
    'testset': 'data/used_cars/test.tsv'
}

used_cars_b1_config = {
    'name': 'UsedCars_B1',
    'task_type': 'classification',
    'vocab': ['0', '1'],
    'trainset': 'data/used_cars/train.tsv',
    'validset': 'data/used_cars/valid.tsv',
    'testset': 'data/used_cars/test_b1_make.tsv'
}

used_cars_b2_config = {
    'name': 'UsedCars_B2',
    'task_type': 'classification',
    'vocab': ['0', '1'],
    'trainset': 'data/used_cars/train.tsv',
    'validset': 'data/used_cars/valid.tsv',
    'testset': 'data/used_cars/test_b2_year.tsv'
}

all_configs.extend([used_cars_config, used_cars_b1_config, used_cars_b2_config])

with open(CONFIGS_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_configs, f, indent=2)

logger.success(f"Task 'UsedCars', 'UsedCars_B1' e 'UsedCars_B2' registrati con successo in {CONFIGS_PATH}:")
logger.info(json.dumps([used_cars_config, used_cars_b1_config, used_cars_b2_config], indent=2))

2026-09-14 15:43:05 | SUCCESS  | Task 'UsedCars', 'UsedCars_B1' e 'UsedCars_B2' registrati con successo in /content/FAIR-DA4ER/ditto/configs.json:
2026-09-14 15:43:05 | INFO     | [
  {
    "name": "UsedCars",
    "task_type": "classification",
    "vocab": [
      "0",
      "1"
    ],
    "trainset": "data/used_cars/train.tsv",
    "validset": "data/used_cars/valid.tsv",
    "testset": "data/used_cars/test.tsv"
  },
  {
    "name": "UsedCars_B1",
    "task_type": "classification",
    "vocab": [
      "0",
      "1"
    ],
    "trainset": "data/used_cars/train.tsv",
    "validset": "data/used_cars/valid.tsv",
    "testset": "data/used_cars/test_b1_make.tsv"
  },
  {
    "name": "UsedCars_B2",
    "task_type": "classification",
    "vocab": [
      "0",
      "1"
    ],
    "trainset": "data/used_cars/train.tsv",
    "validset": "data/used_cars/valid.tsv",
    "testset": "data/used_cars/test_b2_year.tsv"
  }
]


##Addestramento del Modello con `train_ditto.py` (Punto 4.G)

Eseguiamo lo script CLI ufficiale della repository `FAIR-DA4ER`.
Parametri di addestramento:
- **`--lm roberta`**: `roberta-base` (architettura standard di Ditto usata anche nel report PDF di riferimento; in alternativa `distilbert` per maggiore velocità).
- **`--batch_size 16`**: dimensione batch bilanciata per la memoria GPU T4/L4.
- **`--max_len 128`**: lunghezza massima sequenza per i campi automobilistici.
- **`--n_epochs 5`**: 5 epoche di fine-tuning con linear warmup scheduler.
- **`--fp16`**: Mixed Precision nativa PyTorch (GradScaler) per dimezzare il tempo di calcolo.
- **`--save_model`**: salvataggio automatico del miglior checkpoint in `checkpoints/UsedCars/model.pt`.


In [10]:
LM = 'roberta'  # 'roberta' (per accuratezza) oppure 'distilbert' (per velocità)
BATCH_SIZE = 16
MAX_LENGTH = 128
EPOCHS = 5
LR = 3e-5
LOGDIR = 'checkpoints/'
RUN_ID = 42
# RUN_ID = 0 # parametro --run_id (default 0).

cmd = [
    'python', 'train_ditto.py',
    '--task', 'UsedCars',
    '--lm', LM,
    '--batch_size', str(BATCH_SIZE),
    '--max_len', str(MAX_LENGTH),
    '--lr', str(LR),
    '--n_epochs', str(EPOCHS),
    '--fp16',
    '--save_model',
    '--logdir', LOGDIR,
    '--run_id', str(RUN_ID)
]

logger.info(f"Comando di addestramento: {' '.join(cmd)}")
logger.info(f"Directory di lavoro: {REPO_DITTO_DIR}")

train_start = time.perf_counter()
process = subprocess.Popen(
    cmd,
    cwd=str(REPO_DITTO_DIR),
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

for line in process.stdout:
    line_clean = line.rstrip()
    if line_clean:
        logger.info(f"[Ditto Train] {line_clean}")

process.wait()
assert process.returncode == 0, f'Errore durante train_ditto.py (exit code {process.returncode})'

ditto_train_time = time.perf_counter() - train_start

CHECKPOINT_PATH = REPO_DITTO_DIR / LOGDIR / 'UsedCars' / 'model.pt'
assert CHECKPOINT_PATH.exists(), f'Checkpoint non trovato in {CHECKPOINT_PATH}'

subprocess.run(['cp', str(CHECKPOINT_PATH), f'{DITTO_DRIVE_BACKUP}/model.pt'], check=True)

logger.success(f"Addestramento completato in: {ditto_train_time:.2f} s")
logger.info(f"Dimensione checkpoint salvato: {CHECKPOINT_PATH.stat().st_size / (1024*1024):.2f} MB")

2026-09-14 15:43:05 | INFO     | Comando di addestramento: python train_ditto.py --task UsedCars --lm roberta --batch_size 16 --max_len 128 --lr 3e-05 --n_epochs 5 --fp16 --save_model --logdir checkpoints/ --run_id 42
2026-09-14 15:43:05 | INFO     | Directory di lavoro: /content/FAIR-DA4ER/ditto
2026-09-14 15:43:21 | INFO     | [Ditto Train] Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-09-14 15:43:33 | INFO     | [Ditto Train] Running cuda
2026-09-14 15:43:34 | INFO     | [Ditto Train] Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]
2026-09-14 15:43:34 | INFO     | [Ditto Train] Loading weights: 100%|██████████| 197/197 [00:00<00:00, 4619.69it/s]
2026-09-14 15:43:34 | INFO     | [Ditto Train] [transformers] RobertaModel LOAD REPORT from: roberta-base
2026-09-14 15:43:34 | INFO     | [Ditto Train] Key                       | Status     |
2026-09-14 15:43:34 | INFO     | [Ditto Tra

## Caricamento del Checkpoint e Scelta della Soglia 100% via `matcher.tune_threshold()` Ufficiale

Carichiamo il checkpoint del modello addestrato e determiniamo la soglia decisionale ottimale per ciascuna strategia di blocking (B1 e B2) invocando esclusivamente la funzione ufficiale `tune_threshold()` del modulo `matcher.py` di FAIR-DA4ER sui rispettivi validation set dedicati registrati in `configs.json`.


In [11]:
import os
import sys
from pathlib import Path

# Garantisce che la directory di Ditto sia sempre nel path di Python e sia la CWD
REPO_DITTO_DIR = Path('/content/FAIR-DA4ER/ditto')
if not REPO_DITTO_DIR.exists():
    # Fallback se la cartella e diversa
    REPO_DITTO_DIR = Path('/content/ditto')
if str(REPO_DITTO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DITTO_DIR))

os.chdir(str(REPO_DITTO_DIR))

import json
import shutil
import argparse
import torch

from matcher import tune_threshold as matcher_tune_threshold
from ditto_light.ditto import DittoModel

# Caricamento del modello dal checkpoint ufficiale
CHECKPOINT_PATH = REPO_DITTO_DIR / 'checkpoints' / 'UsedCars' / 'model.pt'
if not CHECKPOINT_PATH.exists() and (Path(DITTO_DRIVE_BACKUP) / 'model.pt').exists():
    CHECKPOINT_PATH = Path(DITTO_DRIVE_BACKUP) / 'model.pt'

checkpoint = torch.load(CHECKPOINT_PATH, map_location=device)
model = DittoModel(device=str(device), lm=LM).to(device)
model.load_state_dict(checkpoint['model'])
model.eval()
logger.success(f"Modello Ditto ({LM}) caricato con successo dal checkpoint ufficiale!")

# Validation set SEPARATI per B1 e B2
valid_b1_path = str(DITTO_DATA_DIR / 'valid_b1_make.tsv')
write_ditto_tsv(val_b1_pairs, val_left_ser, val_right_ser, valid_b1_path)

valid_b2_path = str(DITTO_DATA_DIR / 'valid_b2_year.tsv')
write_ditto_tsv(val_b2_pairs, val_left_ser, val_right_ser, valid_b2_path)

# Creazione copie .txt per compatibilita nativa con matcher.py
shutil.copyfile(valid_b1_path, str(DITTO_DATA_DIR / 'valid_b1_make.txt'))
shutil.copyfile(valid_b2_path, str(DITTO_DATA_DIR / 'valid_b2_year.txt'))

subprocess.run(['cp', valid_b1_path, f'{DITTO_DRIVE_BACKUP}/valid_b1_make.tsv'], check=True)
subprocess.run(['cp', valid_b2_path, f'{DITTO_DRIVE_BACKUP}/valid_b2_year.tsv'], check=True)

logger.success("Validation set separati per B1/B2 scritti in formato ufficiale Ditto.")

# Aggiornare configs.json: UsedCars_B1/UsedCars_B2 puntano ognuna al proprio validset dedicato
with open(CONFIGS_PATH, 'r', encoding='utf-8') as f:
    all_configs = json.load(f)

for conf in all_configs:
    if conf['name'] == 'UsedCars_B1':
        conf['validset'] = 'data/used_cars/valid_b1_make.txt'
    elif conf['name'] == 'UsedCars_B2':
        conf['validset'] = 'data/used_cars/valid_b2_year.txt'

with open(CONFIGS_PATH, 'w', encoding='utf-8') as f:
    json.dump(all_configs, f, indent=2)

used_cars_b1_config = next(c for c in all_configs if c['name'] == 'UsedCars_B1')
used_cars_b2_config = next(c for c in all_configs if c['name'] == 'UsedCars_B2')

logger.success("configs.json aggiornato con validset dedicati per B1 e B2:")
logger.info(json.dumps([used_cars_b1_config, used_cars_b2_config], indent=2))

# Tuning soglia tramite la funzione matcher.tune_threshold()
def build_hp(task_name):
    return argparse.Namespace(
        task=task_name,
        lm=LM,
        max_len=MAX_LENGTH,
        summarize=False,
        dk=None
    )

logger.info("Tuning soglia ufficiale (matcher.tune_threshold) per B1 (make)...")
threshold_b1 = matcher_tune_threshold(used_cars_b1_config, model, build_hp('UsedCars_B1'))

logger.info("Tuning soglia ufficiale (matcher.tune_threshold) per B2 (year)...")
threshold_b2 = matcher_tune_threshold(used_cars_b2_config, model, build_hp('UsedCars_B2'))

logger.success(f"Soglia ufficiale B1 (make): {threshold_b1:.4f}")
logger.success(f"Soglia ufficiale B2 (year): {threshold_b2:.4f}")



Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-base
Key                       | Status     | 
--------------------------+------------+-
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


2026-09-14 16:03:53 | SUCCESS  | Modello Ditto (roberta) caricato con successo dal checkpoint ufficiale!
2026-09-14 16:03:53 | INFO     | Salvate 45,548 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/valid_b1_make.tsv
2026-09-14 16:03:53 | INFO     | Salvate 45,771 coppie in: /content/FAIR-DA4ER/ditto/data/used_cars/valid_b2_year.tsv
2026-09-14 16:03:54 | SUCCESS  | Validation set separati per B1/B2 scritti in formato ufficiale Ditto.
2026-09-14 16:03:55 | SUCCESS  | configs.json aggiornato con validset dedicati per B1 e B2:
2026-09-14 16:03:55 | INFO     | [
  {
    "name": "UsedCars_B1",
    "task_type": "classification",
    "vocab": [
      "0",
      "1"
    ],
    "trainset": "data/used_cars/train.tsv",
    "validset": "data/used_cars/valid_b1_make.txt",
    "testset": "data/used_cars/test_b1_make.tsv"
  },
  {
    "name": "UsedCars_B2",
    "task_type": "classification",
    "vocab": [
      "0",
      "1"
    ],
    "trainset": "data/used_cars/train.tsv",
    "validset": "

45548it [03:51, 197.09it/s]


load_f1 = 0.8718282368249838
real_f1 = 0.870843989769821
2026-09-14 16:10:15 | INFO     | Tuning soglia ufficiale (matcher.tune_threshold) per B2 (year)...


45771it [03:51, 197.46it/s]


load_f1 = 0.8734901462174189
real_f1 = 0.8719550281074329
2026-09-14 16:16:38 | SUCCESS  | Soglia ufficiale B1 (make): 0.9500
2026-09-14 16:16:38 | SUCCESS  | Soglia ufficiale B2 (year): 0.9500


## Misurazione Omogenea dell'Inferenza su B1 e B2 (Punto 4.H)

### Dichiarazione Metodologica: Inferenza tramite `matcher.py` ufficiale (FAIR-DA4ER)
L'inferenza viene eseguita sull'intero spazio di coppie candidate generate da B1 (43.456) e B2 (49.757) utilizzando l'interfaccia ufficiale del modulo `matcher.py` della repository:

$$T_{\text{matching}} = T_{\text{model}} + T_{\text{selection}}$$
$$T_{\text{inferenza}} = T_{\text{blocking}} + T_{\text{matching}}$$

1. **$T_{\text{blocking}}$**: Generazione delle candidate pairs dai DataFrame grezzi (`vehicles_test`, `used_cars_test`) tramite blocco B1 (`make`) o B2 (`year`).
2. **$T_{\text{model}}$**: Elaborazione delle coppie mediante forward pass RoBERTa in FP16 su GPU, sincronizzato con `torch.cuda.synchronize()`.
3. **$T_{\text{selection}}$**: Tempo di risoluzione dei conflitti 1:1 mediante ordinamento decrescente per `match_confidence` e deduplicazione bipartita (pari a 0 nella variante **Raw**).
4. **Doppia Rendicontazione**:
   - **Variante Raw**: Output nativo di classificazione pura per soglia da `matcher.py` (senza vincoli 1:1);
   - **Variante 1:1 Selection**: Risoluzione conflitti strutturale ER, direttamente confrontabile con `one_to_one()` di Dedupe e Record Linkage.



In [12]:
import os
import sys
from pathlib import Path

# Garantisce che la directory di Ditto sia sempre nel path e sia CWD
REPO_DITTO_DIR = Path('/content/FAIR-DA4ER/ditto')
if not REPO_DITTO_DIR.exists():
    REPO_DITTO_DIR = Path('/content/ditto')
if str(REPO_DITTO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DITTO_DIR))
os.chdir(str(REPO_DITTO_DIR))

import jsonlines
import time
import numpy as np
import pandas as pd
import torch
from matcher import predict as matcher_predict

def run_homogeneous_inference(pipeline_name, strategy_key, left_df, right_df, gt_df, model, threshold, left_ser, right_ser, lm=LM, max_len=MAX_LENGTH, batch_size=64):
    """
    Pipeline completa di inferenza basata ufficialmente su matcher.py (FAIR-DA4ER):
    1. T_blocking   : candidate generation (identica a Record Linkage e Dedupe)
    2. T_model      : matcher.predict() ufficiale su GPU in FP16 (1 run pulito e preciso)
    3. T_selection  : risoluzione conflitti 1:1 (ordinata per match_confidence decrescente)
    T_matching  = T_model + T_selection
    T_inferenza = T_blocking + T_matching
    """
    # 1. BLOCKING (Candidate Generation)
    t_block_start = time.perf_counter()
    candidate_pairs = candidates_from_blocking(left_df, right_df, strategy_key)
    t_block_end = time.perf_counter()
    blocking_time = t_block_end - t_block_start

    # 2. MATCHING UFFICIALE TRAMITE MATCHER.PY (T_model)
    jsonl_in = DITTO_DATA_DIR / f'candidates_{strategy_key}_input.jsonl'
    jsonl_out = DITTO_DATA_DIR / f'matched_{strategy_key}_output.jsonl'

    with jsonlines.open(jsonl_in, mode='w') as writer:
        for c, u in candidate_pairs[[LEFT_ID, RIGHT_ID]].itertuples(index=False, name=None):
            writer.write([left_ser[c], right_ser[u]])

    def _run_matcher_predict(input_path, output_path):
        if device.type == 'cuda':
            with torch.autocast(device_type='cuda', dtype=torch.float16):
                matcher_predict(
                    input_path=str(input_path),
                    output_path=str(output_path),
                    config=used_cars_config,
                    model=model,
                    batch_size=batch_size,
                    lm=lm,
                    max_len=max_len,
                    threshold=threshold
                )
        else:
            matcher_predict(
                input_path=str(input_path),
                output_path=str(output_path),
                config=used_cars_config,
                model=model,
                batch_size=batch_size,
                lm=lm,
                max_len=max_len,
                threshold=threshold
            )

    # Singolo run cronometrato ad alta precisione con sincronizzazione CUDA
    if device.type == 'cuda':
        torch.cuda.synchronize()
    t_match_start = time.perf_counter()
    _run_matcher_predict(jsonl_in, jsonl_out)
    if device.type == 'cuda':
        torch.cuda.synchronize()
    model_time = time.perf_counter() - t_match_start
    model_time_std = 0.0
    logger.info(f"[{pipeline_name}] Model Time (matcher.py): {model_time:.3f}s")

    # 3. LETTURA PREDIZIONI E CONFIDENZE DA MATCHER.PY
    predictions = []
    confidences = []
    with jsonlines.open(jsonl_out, mode='r') as reader:
        for obj in reader:
            predictions.append(int(obj['match']))
            confidences.append(float(obj.get('match_confidence', 1.0)))

    assert len(predictions) == len(candidate_pairs), (
        f"Disallineamento: {len(predictions)} predizioni vs {len(candidate_pairs)} coppie candidate"
    )

    candidate_eval = candidate_pairs.copy()
    candidate_eval['pred'] = predictions
    candidate_eval['conf'] = confidences

    true_set = set(gt_df[[LEFT_ID, RIGHT_ID]].itertuples(index=False, name=None))

    def compute_eval(pred_set):
        tp = len(pred_set & true_set)
        fp = len(pred_set - true_set)
        fn = len(true_set - pred_set)
        p = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        r = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * p * r / (p + r)) if (p + r) > 0 else 0.0
        return tp, fp, fn, p, r, f1

    # Metriche Raw (decisione pura per soglia da matcher.py, senza risoluzione 1:1)
    predicted_raw = set(
        candidate_eval.loc[candidate_eval['pred'] == 1, [LEFT_ID, RIGHT_ID]]
        .itertuples(index=False, name=None)
    )
    tp_raw, fp_raw, fn_raw, p_raw, r_raw, f1_raw = compute_eval(predicted_raw)
    matching_time_raw = model_time
    total_inference_time_raw = blocking_time + matching_time_raw

    # 4. RISOLUZIONE CONFLITTI 1:1 (ordinata per match_confidence di Ditto)
    t_sel_start = time.perf_counter()
    matches_candidates = candidate_eval[candidate_eval['pred'] == 1].sort_values('conf', ascending=False)
    matches_1to1 = (
        matches_candidates
        .drop_duplicates(subset=[LEFT_ID], keep='first')
        .drop_duplicates(subset=[RIGHT_ID], keep='first')
    )
    selection_time = time.perf_counter() - t_sel_start

    predicted_1to1 = set(matches_1to1[[LEFT_ID, RIGHT_ID]].itertuples(index=False, name=None))
    tp_1to1, fp_1to1, fn_1to1, p_1to1, r_1to1, f1_1to1 = compute_eval(predicted_1to1)

    matching_time_1to1 = model_time + selection_time
    total_inference_time_1to1 = blocking_time + matching_time_1to1

    subprocess.run(['cp', str(jsonl_out), f'{DITTO_DRIVE_BACKUP}/matched_{strategy_key}.jsonl'], check=True)

    logger.info(f"[{pipeline_name}] Raw (Soglia matcher.py)   | TP: {tp_raw:3d} | FP: {fp_raw:3d} | FN: {fn_raw:2d} | Precision: {p_raw:.4f} | Recall: {r_raw:.4f} | F1: {f1_raw:.4f}")
    logger.success(f"[{pipeline_name}] Risoluzione 1:1 (ER)     | TP: {tp_1to1:3d} | FP: {fp_1to1:3d} | FN: {fn_1to1:2d} | Precision: {p_1to1:.4f} | Recall: {r_1to1:.4f} | F1: {f1_1to1:.4f}")
    logger.info(f"[{pipeline_name}] Tempi | Blocking: {blocking_time:.3f}s | Modello: {model_time:.3f}s | Selezione 1:1: {selection_time:.4f}s | Matching(1:1): {matching_time_1to1:.3f}s | Inferenza tot.(1:1): {total_inference_time_1to1:.3f}s")

    return {
        'raw': {
            'Pipeline': f'{pipeline_name} (Raw matcher.py)',
            'Paradigma': 'Deep Learning (Transformer)',
            'Blocking': 'Make' if strategy_key == 'make' else 'Year',
            'Set': 'Test',
            'Candidate Pairs': len(candidate_pairs),
            'TP': tp_raw,
            'FP': fp_raw,
            'FN': fn_raw,
            'Precision': p_raw,
            'Recall': r_raw,
            'F1': f1_raw,
            'Threshold': threshold,
            'Train Time (s)': ditto_train_time,
            'Blocking Time (s)': blocking_time,
            'Model Time (s)': model_time,
            'Model Time Std (s)': model_time_std,
            'Selection Time (s)': 0.0,
            'Matching Time (s)': matching_time_raw,
            'Inference Time (s)': total_inference_time_raw
        },
        '1to1': {
            'Pipeline': f'{pipeline_name}',
            'Paradigma': 'Deep Learning (Transformer)',
            'Blocking': 'Make' if strategy_key == 'make' else 'Year',
            'Set': 'Test',
            'Candidate Pairs': len(candidate_pairs),
            'TP': tp_1to1,
            'FP': fp_1to1,
            'FN': fn_1to1,
            'Precision': p_1to1,
            'Recall': r_1to1,
            'F1': f1_1to1,
            'Threshold': threshold,
            'Train Time (s)': ditto_train_time,
            'Blocking Time (s)': blocking_time,
            'Model Time (s)': model_time,
            'Model Time Std (s)': model_time_std,
            'Selection Time (s)': selection_time,
            'Matching Time (s)': matching_time_1to1,
            'Inference Time (s)': total_inference_time_1to1
        }
    }

logger.info("Avvio inferenza omogenea B1 - Ditto (make) tramite matcher.py...")
res_b1 = run_homogeneous_inference(
    f'B1 - Ditto ({LM})', 'make', vehicles_test, used_cars_test, test_gt, model, threshold_b1,
    test_left_ser, test_right_ser
)

logger.info("Avvio inferenza omogenea B2 - Ditto (year) tramite matcher.py...")
res_b2 = run_homogeneous_inference(
    f'B2 - Ditto ({LM})', 'year', vehicles_test, used_cars_test, test_gt, model, threshold_b2,
    test_left_ser, test_right_ser
)

results_b1_ditto = res_b1['1to1']
results_b2_ditto = res_b2['1to1']

ditto_summary_df = pd.DataFrame([
    res_b1['raw'],
    res_b1['1to1'],
    res_b2['raw'],
    res_b2['1to1']
])
display(ditto_summary_df)



2026-09-14 16:16:38 | INFO     | Avvio inferenza omogenea B1 - Ditto (make) tramite matcher.py...


43456it [18:41, 38.75it/s]

2026-09-14 16:35:20 | INFO     | [B1 - Ditto (roberta)] Model Time (matcher.py): 1121.335s


2026-09-14 16:35:21 | INFO     | [B1 - Ditto (roberta)] Raw (Soglia matcher.py)   | TP: 665 | FP:  74 | FN: 125 | Precision: 0.8999 | Recall: 0.8418 | F1: 0.8698
2026-09-14 16:35:21 | SUCCESS  | [B1 - Ditto (roberta)] Risoluzione 1:1 (ER)     | TP: 651 | FP:   9 | FN: 139 | Precision: 0.9864 | Recall: 0.8241 | F1: 0.8979
2026-09-14 16:35:21 | INFO     | [B1 - Ditto (roberta)] Tempi | Blocking: 0.045s | Modello: 1121.335s | Selezione 1:1: 0.0024s | Matching(1:1): 1121.338s | Inferenza tot.(1:1): 1121.383s
2026-09-14 16:35:21 | INFO     | Avvio inferenza omogenea B2 - Ditto (year) tramite matcher.py...


49757it [21:08, 39.22it/s]


2026-09-14 16:56:31 | INFO     | [B2 - Ditto (roberta)] Model Time (matcher.py): 1270.456s
2026-09-14 16:56:32 | INFO     | [B2 - Ditto (roberta)] Raw (Soglia matcher.py)   | TP: 672 | FP:  78 | FN: 118 | Precision: 0.8960 | Recall: 0.8506 | F1: 0.8727
2026-09-14 16:56:32 | SUCCESS  | [B2 - Ditto (roberta)] Risoluzione 1:1 (ER)     | TP: 657 | FP:   9 | FN: 133 | Precision: 0.9865 | Recall: 0.8316 | F1: 0.9025
2026-09-14 16:56:32 | INFO     | [B2 - Ditto (roberta)] Tempi | Blocking: 0.054s | Modello: 1270.456s | Selezione 1:1: 0.0020s | Matching(1:1): 1270.458s | Inferenza tot.(1:1): 1270.512s


,Pipeline,Paradigma,Blocking,Set,Candidate Pairs,TP,FP,FN,Precision,Recall,F1,Threshold,Train Time (s),Blocking Time (s),Model Time (s),Model Time Std (s),Selection Time (s),Matching Time (s),Inference Time (s)
0,B1 - Ditto (roberta) (Raw matcher.py),Deep Learning (Transformer),Make,Test,43456,665,74,125,0.899865,0.841772,0.869850,0.95,1209.555777,0.045198,1121.335139,0.0,0.000000,1121.335139,1121.380337
1,B1 - Ditto (roberta),Deep Learning (Transformer),Make,Test,43456,651,9,139,0.986364,0.824051,0.897931,0.95,1209.555777,0.045198,1121.335139,0.0,0.002435,1121.337574,1121.382772
2,B2 - Ditto (roberta) (Raw matcher.py),Deep Learning (Transformer),Year,Test,49757,672,78,118,0.896000,0.850633,0.872727,0.95,1209.555777,0.054133,1270.455887,0.0,0.000000,1270.455887,1270.510020
3,B2 - Ditto (roberta),Deep Learning (Transformer),Year,Test,49757,657,9,133,0.986486,0.831646,0.902473,0.95,1209.555777,0.054133,1270.455887,0.0,0.002041,1270.457928,1270.512062


## Conclusione e Salvataggio Risultati di Ditto

I risultati ottenuti da Ditto (confronto tra metriche raw e matching 1-a-1 stabile su B1 e B2) vengono verificati mediante controlli di coerenza formale e salvati come artifact su Google Drive.



In [13]:
#controllo di coerenza: il numero di candidati valutati deve coincidere esattamente con il blocking
assert results_b1_ditto['Candidate Pairs'] == len(test_b1_cand), (
    f"Incoerenza candidati B1: {results_b1_ditto['Candidate Pairs']} vs {len(test_b1_cand)}"
)
assert results_b2_ditto['Candidate Pairs'] == len(test_b2_cand), (
    f"Incoerenza candidati B2: {results_b2_ditto['Candidate Pairs']} vs {len(test_b2_cand)}"
)

#salvataggio su Google Drive dei risultati di Ditto (backup artifact)
ditto_summary_df.to_csv(f'{DITTO_DRIVE_BACKUP}/ditto_results.csv', index=False)
logger.success(f"Risultati di Ditto esportati con successo in: {DITTO_DRIVE_BACKUP}/ditto_results.csv")
logger.info("La comparazione master delle 6 pipeline (Punto 4.H) è gestita nel notebook dedicato HW_6_Confronto_Final.ipynb")

2026-09-14 16:56:34 | SUCCESS  | Risultati di Ditto esportati con successo in: /content/drive/MyDrive/data/ditto_used_cars/ditto_results.csv
2026-09-14 16:56:34 | INFO     | La comparazione master delle 6 pipeline (Punto 4.H) è gestita nel notebook dedicato HW_6_Confronto_Final.ipynb
